In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb
%run stdPlotting.ipynb
os.chdir(d['py'])
import mCGE

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.
The file _gams_py_gdb2.gdx is still active and was not deleted.
The file _gams_py_gdb3.gdx is still active and was not deleted.
The file _gams_py_gdb4.gdx is still active and was not deleted.
The file _gams_py_gdb5.gdx is still active and was not deleted.
The file _gams_py_gdb6.gdx is still active and was not deleted.
The file _gams_py_gdb7.gdx is still active and was not deleted.


In [2]:
d['figs'] = os.path.join(d['curr'],'results')
os.chdir(d['py'])
import mCGE 
os.chdir(os.path.join(d['curr'], 'py'))
import report

# Plots

Load model + data from shocks:

In [3]:
# model:
t0 = 2019
name = f'vGRSIntRC{t0}CGE'
M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
ws = M.ws 
# Data:
with open(os.path.join(d['curr'],'results','GRSIntRC_shocks'), "rb") as file:
    shocks = pickle.load(file)

Settings for plotting:

In [4]:
slides = True
tPlot_1 = 2030 # if one year in plot, use this
tPlot = pd.Index(range(t0, 2031), name = 't') # if multiple years, use this

Experiments names:

In [5]:
shockNames = {'Baseline': 'Baseline',
              'IRRall': 'Experiment 1',
              'IRRplastic': 'Experiment 2',
              'RCEff': 'Experiment 3',
              'taxVirgin': 'Experiment 4', 
              'taxWasteGen': 'Experiment 5',
              'RCmandate': 'Experiment 6',
              'subsidyRecycledInputs': 'Experiment 7'}

## 1. Compare CR and virgin use

Plot total CR across experiments:

In [6]:
CR_tot = pd.Series({shockNames[k]: shocks[k]['CircularRateTot'].xs(tPlot_1) for k,v in shockNames.items()})
ΔCR_tot = CR_tot[1:]-CR_tot['Baseline']

*Version 1:* Circularity rate with baseline as solid line.

In [7]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8));
CR_tot[1:].plot.bar(ax = ax);
ax.axhline(CR_tot['Baseline'], color = 'k', linewidth = 1.5);
ax.set_ylabel('Circularity rate');
fig.tight_layout();
fig.savefig(os.path.join(d['figs'],f"CRTotAll_v1.pdf"),edgecolor='k')

*Version 2:* Change in CR:

In [8]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8));
ΔCR_tot.plot.bar(ax = ax);
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
ax.set_ylabel('Δ Circularity rate');
fig.tight_layout();
fig.savefig(os.path.join(d['figs'],f"CRTotAll_v2.pdf"),edgecolor='k')

Plot relative change in virgin materials in a similar plot:

In [9]:
ΔQv_tot = pd.Series({shockNames[k]: shocks[k]['ΔQvTot_QvTot'].xs(tPlot_1) for k,v in shockNames.items() if k != 'Baseline'})

In [10]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8));
ΔQv_tot.plot.bar(ax = ax);
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
ax.set_ylabel('$\\Delta Q_v / Q_v$');
fig.tight_layout();
fig.savefig(os.path.join(d['figs'],f"DeltaQvTotAll_v2.pdf"),edgecolor='k')

## 2. Prelimenary look at the data

### A. Share of demand with substitution between recycled/virgin

In [11]:
Rep = report.Standard(shocks['Baseline']['db'])

Nesting structure for all relevant producers/consumers:

In [12]:
m = reduce(pd.Index.union, [M.get('map', m = m) for m in ('P','W','C','G')])

Identify nodes that actually combines recycled/virgin:

In [13]:
m_ZOnm = m[m.get_level_values('n').isin('RxE_'+M.db('m'))].droplevel('nn').unique() # parent nodes in nesting tree that combines virgin and recycled

Map to $m$ type:

In [14]:
mapToM = pd.MultiIndex.from_arrays(['RxE_'+M.db('m'), M.db('m')], names = ['n','m'])
m_ZOm = adjMultiIndex.applyMult(m_ZOnm, mapToM).droplevel('n')

Map to relevant final good types:

In [15]:
m_ZOn = adjMultiIndex.applyMult(m_ZOm, Rep.n2m)

Material demand covered by industries that rely both on virgin/recycled goods:

In [16]:
qD_RV = adjMultiIndex.applyMult(adj.rc_pd(M.db('qD').xs(t0), m_ZOn), Rep.n2m).groupby(['m']).sum()

Compare to total demand:

In [17]:
qD_Tot = shocks['Baseline']['Qt'].xs(t0)
qD_RV/qD_Tot

m
Metal      0.949833
Paper      0.832628
Plastic    0.867402
Rubber     0.427689
Textile         NaN
dtype: float64

### B. How is the increase in demand for recycled materials spread across industries?

What industries increase their demand for recycled plastics?

In [18]:
qD_R = adj.rc_pd(shocks['RCEff']['db']('qD'), Rep.rm).xs(t0)
qD_RShock = adjMultiIndex.applyMult(qD_R, Rep.n2m).groupby(['s','m']).sum()

Note: Recycled plastic is exported much less than other material types. Thus, in our measure of secondary material use, plastic increases less. 

In [19]:
qD_R = adj.rc_pd(shocks['Baseline']['db']('qD'), Rep.rm).xs(t0)
qD_RBase = adjMultiIndex.applyMult(qD_R, Rep.n2m).groupby(['s','m']).sum()
(qD_RShock-qD_RBase).unstack('m')

m,Metal,Paper,Plastic,Rubber,Textile
s,,,,,
Energy,0.000030,NaN,NaN,NaN,NaN
F,0.073454,0.017085,0.012036,0.023037,NaN
I_K,0.000272,0.000121,NaN,NaN,NaN
Waste,0.000129,0.000159,NaN,NaN,0.000168
man_met,0.002293,NaN,NaN,NaN,NaN
man_oth,0.000747,NaN,0.007220,NaN,NaN
man_rub_pla,0.000076,0.001943,0.069168,0.001054,NaN
other,0.000058,NaN,NaN,NaN,0.000001
sale,0.002124,NaN,NaN,NaN,NaN


## Experiment 1 - Improvement of internal recycling efficiency for all materials

**Short description:** The internal recycling efficiency is increased by 10 percentage points for all materials simultaneously. 

Total rebound effect over time:

In [20]:
shockId = 'IRRall'
di = shocks[shockId]

In [21]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [22]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp1_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [23]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [24]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [25]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [26]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [27]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [28]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [29]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_rp.pdf"),edgecolor='k')

## Experiment 2 -  Improvement of internal recycling efficiency for one material type

**Short description:** The internal recycling efficiency is increased by 10 percentage points for one material type. This is done for plastics and metal separately. 

In [30]:
di_all = shocks['IRRall']
di_plastic = shocks['IRRplastic']
di_metal = shocks['IRRmetal']

Compare rebound effects of single material type vs. all:

*For metal:*

In [31]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(pd.concat([di_all['Rbv'].xs('Metal',level='m').rename('All'), 
                                             di_metal['Rbv'].xs('Metal',level='m').rename('Metal only')], axis = 1), tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_ReboundMetal.pdf"),edgecolor='k')

*For plastic:*

In [32]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(pd.concat([di_all['Rbv'].xs('Plastic',level='m').rename('All'), 
                                             di_plastic['Rbv'].xs('Plastic',level='m').rename('Plastic only')], axis = 1), tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_ReboundPlastic.pdf"),edgecolor='k')

**Add "standard" plots for the plastic shocks:** 

Total rebound effect over time:

In [33]:
shockId = 'IRRplastic'
di = shocks[shockId]

In [34]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [35]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp2_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [36]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [37]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [38]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [39]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [40]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [41]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [42]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_rp.pdf"),edgecolor='k')

## Experiment 3 -  Improvement of internal recycling efficiency for one material type

**Short description:** The recycling efficiency in the solid waste management sector is increased for all materials simultaneously. 

Total rebound effect over time:

In [43]:
shockId = 'RCEff'
di = shocks[shockId]

In [44]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [45]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp3_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [46]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [47]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [48]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [49]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [50]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [51]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [52]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_rp.pdf"),edgecolor='k')

## Experiment 4 -  Tax on virgin material use

**Short description:** An ad-valorem tax of 10 percentage points is introduced on all virgin materials. 

Total rebound effect over time:

In [53]:
shockId = 'taxVirgin'
di = shocks[shockId]

In [54]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [55]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp4_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [56]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [57]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [58]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [59]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [60]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [61]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [62]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_rp.pdf"),edgecolor='k')

## Experiment 5 - Waste tax

**Short description:** A tax on waste is introduced for all sectors and all waste types. The tax is based on mass. 

Total rebound effect over time:

In [63]:
shockId = 'taxWasteGen'
di = shocks[shockId]

In [64]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [65]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp5_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [66]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [67]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [68]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [69]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [70]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [71]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [72]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_rp.pdf"),edgecolor='k')

Change in *marginal effective cost* of using virgin materials:

In [73]:
def getVm(db, name):
    x = adj.rc_pd(db(name).xs(tPlot_1), Rep.vm)
    x = adj.rc_pd(x, ('not', db('s_f'))) # remove foreign sector deman
    return adjMultiIndex.applyMult(x, Rep.n2m)

Average effective cost of virgin materials:

In [74]:
qD_vm = getVm(shocks['Baseline']['db'], 'qD')
pD_vm = getVm(shocks['Baseline']['db'], 'pD')
avgP = (qD_vm * pD_vm).groupby('m').sum() / qD_vm.groupby('m').sum()

After the shock:

In [75]:
qDshock = getVm(di['db'], 'qD')
pDshock = getVm(di['db'], 'pD')
avgPshock = (qDshock * pDshock).groupby('m').sum() / qDshock.groupby('m').sum()

## Experiment 6 - Mandate to increase recycling rate

**Short description:** A mandate to increase recycling of textiles in the solid waste management sector.

Total rebound effect over time:

In [76]:
shockId = 'RCmandate'
di = shocks[shockId]

In [77]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [78]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp6_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [79]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [80]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [81]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [82]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [83]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [84]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [85]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_rp.pdf"),edgecolor='k')

## Experiment 7 - Subsidy to recycled materials

**Short description:** Introducing a subsidy to recycled material use in all production sectors. 

Total rebound effect over time:

In [86]:
shockId = 'subsidyRecycledInputs'
di = shocks[shockId]

In [87]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [88]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp7_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [89]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [90]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [91]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [92]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [93]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [94]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [95]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_rp.pdf"),edgecolor='k')